In [1]:
#loading metacells of rna and atac data
import anndata
from anndata import AnnData
import pandas as pd
import scanpy as sc

sc_rna = anndata.read_h5ad("/home/fgsasse_lrs_1/Downloads/BA/BA_data/SEAcells/SEACell_summarized_annot_RNA.h5ad")  
sc_atac = anndata.read_h5ad("/home/fgsasse_lrs_1/Downloads/BA/BA_data/SEAcells/SEACell_summarized_annot_ATAC.h5ad")

gene_peaks_10kb = pd.read_csv("/home/fgsasse_lrs_1/Downloads/BA/BA_data/TSS windows/gene_peak_assignments_10kb.csv")
gene_peaks_20kb = pd.read_csv("/home/fgsasse_lrs_1/Downloads/BA/BA_data/TSS windows/gene_peak_assignments_20kb.csv")
gene_peaks_50kb = pd.read_csv("/home/fgsasse_lrs_1/Downloads/BA/BA_data/TSS windows/gene_peak_assignments_50kb.csv")
gene_peaks_100kb = pd.read_csv("/home/fgsasse_lrs_1/Downloads/BA/BA_data/TSS windows/gene_peak_assignments_100kb.csv")

In [2]:
#function for normalizing the aggregated data by total counts per SEACell to get relative accessibility/expression values (compositional normalization)
def compositional_normalize_adata(adata):
    import pandas as pd
    import scipy.sparse as sp
    
    # extract to dense DataFrame
    if sp.issparse(adata.X):
        df = pd.DataFrame(adata.X.toarray(), index=adata.obs_names, columns=adata.var_names)
    else:
        df = pd.DataFrame(adata.X, index=adata.obs_names, columns=adata.var_names)
    
    # normalize
    row_sums = df.sum(axis=1)
    if (row_sums == 0).any():
        print(f"Warning: {(row_sums == 0).sum()} metacells with zero counts.")
    df_norm = df.div(row_sums, axis=0)
    
    # put back into AnnData, preserving obs/var metadata
    adata_norm = adata.copy()
    adata_norm.X = df_norm.values
    return adata_norm

sc_rna_norm = compositional_normalize_adata(sc_rna)
sc_atac_norm = compositional_normalize_adata(sc_atac)

In [3]:
#check dimensions of the data
print(sc_atac_norm.X.shape)
print(sc_rna_norm.X.shape)

print(sc_atac_norm.obs_names)
print(sc_rna_norm.obs_names)

(1260, 640834)
(1260, 19380)
Index(['SEACell-1201', 'SEACell-667', 'SEACell-259', 'SEACell-541',
       'SEACell-503', 'SEACell-109', 'SEACell-478', 'SEACell-241',
       'SEACell-1045', 'SEACell-624',
       ...
       'SEACell-742', 'SEACell-1009', 'SEACell-941', 'SEACell-294',
       'SEACell-1224', 'SEACell-983', 'SEACell-1055', 'SEACell-1119',
       'SEACell-543', 'SEACell-665'],
      dtype='object', length=1260)
Index(['SEACell-1201', 'SEACell-667', 'SEACell-259', 'SEACell-541',
       'SEACell-503', 'SEACell-109', 'SEACell-478', 'SEACell-241',
       'SEACell-1045', 'SEACell-624',
       ...
       'SEACell-742', 'SEACell-1009', 'SEACell-941', 'SEACell-294',
       'SEACell-1224', 'SEACell-983', 'SEACell-1055', 'SEACell-1119',
       'SEACell-543', 'SEACell-665'],
      dtype='object', length=1260)


In [4]:
#Reindex the dataframes to ensure they are in the same order
sc_atac_norm = sc_atac_norm[sc_rna_norm.obs_names, :]
sc_rna_norm = sc_rna_norm[sc_atac_norm.obs_names, :]
print(sc_atac_norm.obs_names.equals(sc_rna_norm.obs_names))  # Should return True
sc_rna_norm.shape

True


(1260, 19380)

In [5]:
#Substract the genes that are present in the gene_peaks_10kb dataframe from the sc_rna_norm dataframe, to only keep the genes that have peaks assigned to them
genes_with_peaks_10kb = gene_peaks_10kb["gene_id"].tolist()
sc_rna_norm = sc_rna_norm[:, genes_with_peaks_10kb] 
print(sc_rna_norm.shape)

(1260, 19380)


In [6]:
print(sc_atac_norm.X)
print(sc_rna_norm.X)

[[0.00000000e+00 4.71352320e-06 0.00000000e+00 ... 0.00000000e+00
  2.57615401e-05 2.52439246e-05]
 [0.00000000e+00 2.68460466e-06 0.00000000e+00 ... 2.68460466e-06
  2.24113381e-05 2.18475583e-05]
 [0.00000000e+00 0.00000000e+00 2.87455094e-06 ... 3.65690981e-06
  2.09852149e-05 2.04339089e-05]
 ...
 [0.00000000e+00 0.00000000e+00 3.64139948e-06 ... 0.00000000e+00
  2.59808564e-05 2.57897504e-05]
 [2.56903260e-06 0.00000000e+00 4.86367987e-06 ... 0.00000000e+00
  2.24839954e-05 2.23332046e-05]
 [0.00000000e+00 4.42427346e-06 0.00000000e+00 ... 0.00000000e+00
  2.46272639e-05 2.45654893e-05]]
[[2.89902113e-04 4.17561525e-05 0.00000000e+00 ... 4.17561525e-05
  2.57144122e-05 1.83789940e-04]
 [2.51422939e-04 5.36276609e-05 3.52043969e-05 ... 0.00000000e+00
  9.54403988e-05 1.40350210e-04]
 [2.44855780e-04 7.73136448e-05 0.00000000e+00 ... 2.75135157e-05
  0.00000000e+00 1.54726141e-04]
 ...
 [3.42530986e-04 1.20932344e-04 0.00000000e+00 ... 0.00000000e+00
  0.00000000e+00 2.33675377e-04]

In [7]:
#taking the minimum non-zero value in the sc_rna_norm matrix to add it to all values before log transformation to avoid taking log of zero
import numpy as np
non_zero_mask = (sc_rna_norm.X > 0)
epsilon_rna = np.min(sc_rna_norm.X[non_zero_mask])

#taking the minimum non-zero value in the sc_atac_norm matrix to add it to all values before log transformation to avoid taking log of zero
non_zero_mask = (sc_atac_norm.X > 0)
epsilon_at= np.min(sc_atac_norm.X[non_zero_mask])

In [8]:
#log transformation (for norm. distribution) & scaling 10000000+1
#log scaling both datasets by multiplying by 10 million and adding 1 to avoid log(0) issues, then taking log10
sc_rna_norm.X = np.log10((sc_rna_norm.X+ epsilon_rna)*10000000) 
sc_atac_norm.X = np.log10((sc_atac_norm.X+ epsilon_at)*10000000)
print(sc_atac_norm.X)
print(sc_rna_norm.X)

/home/fgsasse_lrs_1/miniforge3/envs/seacells/lib/python3.12/site-packages/anndata/_core/anndata.py:639: FutureWarning: Setting element `.X` of view of `AnnData` object will obey copy-on-write semantics in the next minor release. 
  if self._handle_view_X_cow(value):
/tmp/ipykernel_3638352/3560636608.py:3: ImplicitModificationWarning: Modifying `X` on a view results in data being overridden
  sc_rna_norm.X = np.log10((sc_rna_norm.X+ epsilon_rna)*10000000)
/tmp/ipykernel_3638352/3560636608.py:4: ImplicitModificationWarning: Modifying `X` on a view results in data being overridden
  sc_atac_norm.X = np.log10((sc_atac_norm.X+ epsilon_at)*10000000)


[[0.43982004 1.69799888 0.43982004 ... 0.43982004 2.41558842 2.40686762]
 [0.43982004 1.47127901 0.43982004 ... 1.47127901 2.35577031 2.34484141]
 [0.43982004 0.43982004 1.49829121 ... 1.59463765 2.32757396 2.31616369]
 ...
 [0.43982004 0.43982004 1.59292122 ... 0.43982004 2.41923129 2.41605872]
 [1.45398172 0.43982004 1.71087766 ... 0.43982004 2.35715898 2.35427201]
 [0.43982004 1.67205925 0.43982004 ... 0.43982004 2.39624422 2.39516554]]
[[3.4748828  2.70166939 1.93225017 ... 2.70166939 2.53491418 3.28408211]
 [3.41493749 2.79367345 2.64107722 ... 1.93225017 3.01701661 3.17291163]
 [3.4038261  2.93383761 1.93225017 ... 2.55713542 1.93225017 3.21293761]
 ...
 [3.54541423 3.11222931 1.93225017 ... 1.93225017 1.93225017 3.38422967]
 [3.47596518 3.04327385 1.93225017 ... 1.93225017 2.92368264 3.30750865]
 [3.58442372 1.93225017 1.93225017 ... 1.93225017 1.93225017 3.39563446]]


In [ ]:
#Plotting the distribution of the value of the genes across all cell types in a boxplot for RNA data
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

P_rna_scaled = np.log10((sc_rna_norm.X+ epsilon_rna)*10000000) 

# RNA counts x samples
P_rna_scaled = P_rna_scaled.T.astype(float)

# convert to long format for seaborn
P_rna_scaled_plot = pd.DataFrame(P_rna_scaled, columns=sc_rna_norm.obs_names).melt(
	var_name="celltype",
	value_name="log_cpm"
)
#Plotting the distribution of values of the genes across all cell types in a density plot for RNA data
plt.figure(figsize=(10, 5))
sns.kdeplot(data=P_rna_scaled_plot, x="log_cpm", common_norm=False, fill=True, alpha=0.5)
plt.xlabel("log10((RNA + min(RNA)) * 10000000)")
plt.title("Density of log10((RNA + min(RNA)) * 10000000) across cell type")
plt.tight_layout()
plt.show()

In [ ]:
#taking the minimum non-zero value in the sc_atac_norm matrix to add it to all values before log transformation to avoid taking log of zero
non_zero_mask = (sc_atac_norm.X > 0)
epsilon_at= np.min(sc_atac_norm.X[non_zero_mask])

In [ ]:
#Plotting the distribution of the value of the peaks across all cell types in a boxplot for ATAC data
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

P_atac_scaled = np.log10((sc_atac_norm.X+ epsilon_at)*10000000) 

# ATAC counts x samples
P_atac_scaled = P_atac_scaled.T.astype(float)

# convert to long format for seaborn
P_atac_scaled_plot = pd.DataFrame(P_atac_scaled, columns=sc_atac_norm.obs_names).melt(
	var_name="celltype",
	value_name="log_cpm"
)
#Plotting the distribution of values of the peaks across all cell types in a density plot for ATAC data
plt.figure(figsize=(10, 5))
sns.kdeplot(data=P_atac_scaled_plot, x="log_cpm", common_norm=False, fill=True, alpha=0.5)
plt.xlabel("log10((ATAC + min(ATAC)) * 10000000)")
plt.title("Density of log10((ATAC + min(ATAC)) * 10000000) across cell type")
plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import sys

repo_root = Path("/home/fgsasse_lrs_1/Downloads/BA")
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from your_package.correlation import (
    DEFAULT_WINDOW_LABELS,
    add_correlation_categories,
    aggregate_correlation_categories,
    build_correlation_dataframe,
    compute_all_window_peak_correlations,
)

window_assignments = {
    "10kb": gene_peaks_10kb,
    "20kb": gene_peaks_20kb,
    "50kb": gene_peaks_50kb,
    "100kb": gene_peaks_100kb,
}

all_window_results = compute_all_window_peak_correlations(
    atac_data=sc_atac_norm,
    rna_data=sc_rna_norm,
    window_assignments=window_assignments,
)

cor_res_df = build_correlation_dataframe(
    all_window_results,
    window_labels=DEFAULT_WINDOW_LABELS,
)

#save the correlation results as a csv file
cor_res_df.to_csv("/home/fgsasse_lrs_1/Downloads/BA/BA_data/Correlations/Results/gene_peak_correlation_sc_results.csv", index=False)

cor_res_df = add_correlation_categories(cor_res_df)
agg_cor_df = aggregate_correlation_categories(cor_res_df)


print(cor_res_df.shape)
print(cor_res_df.head())
print(agg_cor_df.head())

# Dedublication
of gene-peak pairs, meaning we keep only the first occurrence of each pair across all windows
- ordering by window size, we keep the pair with the smallest window size

In [ ]:
# classify every peak–gene pair into one of three categories 
def classify_pair(row):
    if row["padj"] <= 0.05 and row["correlation"] < 0:
        return "sig. negative"
    elif row["padj"] <= 0.05 and row["correlation"] > 0:
        return "sig. positive"
    else:
        return "non-significant"

cor_res_df["category"] = cor_res_df.apply(classify_pair, axis=1)

# Make window categorical and ordered
cor_res_df["window"] = pd.Categorical(
    cor_res_df["window"],
    categories=["10kb", "20kb", "50kb", "100kb"],
    ordered=True
)

# Order by window
cor_res_df = cor_res_df.sort_values("window", ascending=True)
cor_res_df

cor_res_df["category"] = cor_res_df.apply(classify_pair, axis=1)

# "Dedublication" of gene-peak pairs by keeping only the first occurrence of each pair across all windows (since they are ordered by window size, this will keep the pair with the smallest window size)"
pair_summary = (
    cor_res_df
    .groupby(["gene", "peak"], as_index=False)
    .first()
)

# Sort by window 
pair_summary = pair_summary.sort_values("window", ascending=True)
pair_summary

# Sanity check - how many times each gene-peak pair appears in the summary
pair_counts = pair_summary.groupby(["gene", "peak"]).size().reset_index(name="count")
pair_counts['count'].value_counts()

# Counts how many unique gene-peak pairs fall into each category for each window size
summary_counts = pair_summary.groupby(['window', 'gene', 'category']).size().reset_index(name="count")
summary_counts


In [ ]:
#Boxplot of the counts of peaks per gene (first occurrence of gene-peak pair across all windows) for each category and window size
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

category_order  = ["sig. negative", "non-significant", "sig. positive"]
category_colors = {
    "sig. negative":   "#D43737",
    "sig. positive":   "#1A8EB9",
    "non-significant": "#A9A9A9",
}

fig, ax = plt.subplots(figsize=(9, 6))
fig.patch.set_facecolor("white")
ax.set_facecolor("#F8F8F8")

sns.boxplot(
    data=summary_counts,
    x="window", y="count", hue="category",
    hue_order=category_order,
    palette=category_colors,
    width=0.6,
    linewidth=1.2,
    fill=True,
    flierprops=dict(marker="o", markersize=4, linewidth=0),
    boxprops=dict(alpha=1),
    gap=0.1,
    ax=ax,
)

# Replace x-axis tick labels
window_labels = {
    "10kb":  "±TSS–10 kb",
    "20kb":  "±10–20 kb",
    "50kb":  "±20–50 kb",
    "100kb": "±50–100 kb",
}
ax.set_xticklabels([window_labels[t.get_text()] for t in ax.get_xticklabels()])

# Recolor fliers manually to match their box color
for line, color in zip(
    [c for c in ax.get_lines() if c.get_linestyle() == "none"],
    [category_colors[cat]
     for _ in summary_counts["window"].cat.categories
     for cat in category_order]
):
    line.set_markerfacecolor(color)
    line.set_markeredgecolor(color)
    line.set_alpha(0.7)

ax.set_title("Peak counts per gene across genomic windows (ct + time)",
             fontsize=14, fontweight="500", pad=14, loc="center")
ax.set_xlabel("Genomic window", fontsize=11, labelpad=8)
ax.set_ylabel("# significant peaks per gene", fontsize=11, labelpad=8)

ax.yaxis.grid(True, color="#cccccc", linewidth=0.8, linestyle="-", zorder=0)
ax.xaxis.grid(False)
ax.set_axisbelow(True)

for spine in ax.spines.values():
    spine.set_color("#a5a4a4")

handles = [mpatches.Patch(color=category_colors[c], alpha=1, label=c.capitalize())
           for c in category_order]
ax.legend(
    handles=handles,
    title="Correlation Category\n(padj ≤ 0.05)",
    frameon=True,
    fontsize=10,
    loc="upper left",
    title_fontsize=10,
)

plt.tight_layout()
plt.show()

In [ ]:
# Boxplot of the counts of peaks per gene (first occurrence of gene-peak pair across all windows) for significant categories only (non-significant category excluded) for each window size
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

category_order  = ["sig. negative", "sig. positive"]
category_colors = {
    "sig. negative":   "#D43737",
    "sig. positive":   "#1A8EB9",
}

only_sig = summary_counts[summary_counts["category"] != "non-significant"]

fig, ax = plt.subplots(figsize=(9, 6))
fig.patch.set_facecolor("white")
ax.set_facecolor("#F8F8F8")

sns.boxplot(
    data=only_sig,
    x="window", y="count", hue="category",
    hue_order=category_order,
    palette=category_colors,
    width=0.6,
    linewidth=1.2,
    fill=True,
    flierprops=dict(marker="o", markersize=4, linewidth=0),
    boxprops=dict(alpha=1),
    gap=0.1,
    ax=ax,
)

# Replace x-axis tick labels
window_labels = {
    "10kb":  "±TSS–10 kb",
    "20kb":  "±10–20 kb",
    "50kb":  "±20–50 kb",
    "100kb": "±50–100 kb",
}
ax.set_xticklabels([window_labels[t.get_text()] for t in ax.get_xticklabels()])

# Recolor fliers manually to match their box color
for line, color in zip(
    [c for c in ax.get_lines() if c.get_linestyle() == "none"],
    [category_colors[cat]
     for _ in only_sig["window"].cat.categories
     for cat in category_order]
):
    line.set_markerfacecolor(color)
    line.set_markeredgecolor(color)
    line.set_alpha(0.7)

ax.set_title("Peak counts per gene across genomic windows (ct + time)",
             fontsize=14, fontweight="500", pad=14, loc="center")
ax.set_xlabel("Genomic window", fontsize=11, labelpad=8)
ax.set_ylabel("# significant peaks per gene", fontsize=11, labelpad=8)

ax.yaxis.grid(True, color="#cccccc", linewidth=0.8, linestyle="-", zorder=0)
ax.xaxis.grid(False)
ax.set_axisbelow(True)

for spine in ax.spines.values():
    spine.set_color("#a5a4a4")

handles = [mpatches.Patch(color=category_colors[c], alpha=1, label=c.capitalize())
           for c in category_order]
ax.legend(
    handles=handles,
    title="Correlation Category\n(padj ≤ 0.05)",
    frameon=True,
    fontsize=10,
    loc="upper left",
    title_fontsize=10,
)

plt.tight_layout()
plt.show()